## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](atchitecture_diagram.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

----------------------------------------------------------------------------------------------------------------------------------------
#### ANSWER

##### Interrelationships Between the Three States

The states form a hierarchical delegation pattern:

1. **AgentState** (top-level) → Contains the full workflow context including user messages, research brief, accumulated notes, and final report
2. **SupervisorState** (middle) → Manages research delegation with supervisor-specific messages, tracks research iterations, and accumulates notes from multiple researchers
3. **ResearcherState** (bottom) → Handles individual research tasks with their own message threads and tool call iterations

Data flows down when delegating work (AgentState → SupervisorState → ResearcherState) and up when returning results (ResearcherState outputs compressed_research and raw_notes back to SupervisorState, which eventually updates AgentState).

##### Why Not One Huge State?

Separation provides critical benefits:

1. **Scope isolation** - Each component only accesses data it needs, preventing accidental cross-contamination
2. **Parallel execution** - Multiple researchers can operate simultaneously with independent state (their own researcher_messages and tool_call_iterations) without conflicts
3. **Clear contracts** - Type definitions enforce what data flows between levels (e.g., ResearcherOutputState explicitly defines what researchers return)
4. **Memory efficiency** - Researchers don't carry the entire conversation history, just their specific research thread
5. **Maintainability** - Easier to debug and modify when each level has well-defined responsibilities

A single huge state would create a tangled mess where all components could interfere with each other's data and make parallel execution difficult.

----------------------------------------------------------------------------------------------------------------------------------------


## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

----------------------------------------------------------------------------------------------------------------------------------------
### ANSWER

#### Advantages and Disadvantages of Importing Components

##### Advantages of Importing from Library

1. **Maintainability** - Bug fixes and improvements happen in one place; all notebooks benefit automatically

2. **Code reusability** - Share complex logic (like the 7-step `tavily_search` with async summarization) across multiple notebooks without duplication

3. **Cleaner notebooks** - Focus on research workflow logic rather than implementation details. The notebook shows *what* you're doing, the library shows *how*

4. **Testability** - Utilities can be unit tested independently, ensuring reliability

5. **Modularity** - Clear separation between infrastructure code (API calls, error handling, token management) and business logic (research workflow)

6. **Professional structure** - Follows software engineering best practices for production systems

---

##### Disadvantages of Importing from Library

1. **Reduced visibility** - Can't see implementation without navigating to source files. For learning, this hides important details (e.g., how `tavily_search` handles rate limits or summarization)

2. **Harder to experiment** - Modifying behavior requires editing the library and reloading, not just tweaking a cell

3. **Dependency management** - Notebook isn't self-contained; requires library installation and correct version

4. **Debugging complexity** - Stack traces span multiple files; harder to step through code during development

5. **Version coupling** - Notebook may break if library API changes; need version pinning

6. **Context switching** - Understanding full behavior requires jumping between notebook and library files


----------------------------------------------------------------------------------------------------------------------------------------

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [18]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with the analysis. You've provided a comprehensive PDF document from the NBER Working Paper "How People Use ChatGPT" and requested insights on:

1. Main findings about how people are using AI (specifically ChatGPT)
2. Most common use cases 
3. Trends and patterns from the data

The document contains substantial data from ChatGPT usage between May 2024 and June 2025, including message classifications, user demographics, and usage patterns. I will now analyze this research paper and provide detailed insights on all three requested areas based on the findings presented in the document.

Node: write_research_brief

Research Brief Generated:
I have provided a comprehensive NBER Working Paper titled "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Chatterji et al. that analyzes ChatGPT usage patterns from May 2024 to June 2025. I need you to thoroughly analyze this document 


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# How People Use ChatGPT: Comprehensive Analysis from NBER Research

## Overview and Research Scope

The NBER Working Paper "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Chatterji et al. represents the first comprehensive study of ChatGPT consumer usage patterns, analyzing data from May 2024 through June 2025. This groundbreaking research documents ChatGPT's explosive growth from its November 2022 launch to become one of the fastest-adopted technologies in history, reaching approximately 10% of the world's adult population by July 2025.[1][2][3]

The study employed unprecedented privacy protection measures using a Data Clean Room approach where researchers never directly accessed user data or personal information. All analyses were conducted on automatically anonymized messages, with researchers only receiving aggregated outputs across limited categories, setting "a new precedent and very high bar for privacy protection."[2] The research is based on three primary datasets: total daily message volumes, approximately one million classified messages, and aggregated employment/education data for 130,000 users.[5]

## Main Findings About ChatGPT Usage Patterns and Demographics

### Unprecedented Adoption Rates and Growth

By July 2025, ChatGPT had achieved remarkable scale with more than 700 million weekly active users—representing about 10% of the world's adult population.[1][2][3] The platform's growth trajectory has been extraordinary: reaching 1 million users by December 5, 2022, hitting 100 million weekly active users in November 2023, and doubling users every 7-8 months since launch.[2]

As of June 2025, ChatGPT users were sending more than 2.6 billion messages per day, or more than 30,000 messages per second.[2] Total message volume increased by 5.8x in just one year, growing from 451 million daily messages in June 2024 to 2,627 million in June 2025. By July 2025, the platform was processing 18 billion messages per week.[7]

The speed of adoption significantly outpaces historical technology adoption patterns. ChatGPT reached 1 billion messages in December 2024, less than two years after release, compared to Google Search which took eight years to reach 1 billion daily searches after its 1999 public launch.[2]

### Demographic Distribution and Shifts

**Age Demographics**: Nearly half of all messages sent by adults come from users under age 26, with 46% of users in the study sample falling between ages 18-25.[1][3] Users aged 18-25 generate 46% of total messages, though work-related usage is higher among users aged 30-60.[10]

**Gender Evolution**: The research reveals a dramatic closing of the gender gap. Initially, over 80% of users had typically masculine first names, but this shifted significantly over time. Among users with classifiable names, the percentage with typically feminine names grew from 37% in January 2024 to 52% by July 2025, indicating full gender parity had been achieved.[1][2][6]

**Geographic and Economic Patterns**: Usage growth has been fastest in lower-income countries, with growth rates in the lowest income countries over 4x those in the highest income countries by May 2025. Middle-income countries ($10,000-$40,000 GDP per capita) showed the highest growth rates between May 2024 and May 2025. Countries at the 50th percentile of GDP per capita now show similar usage rates to those at the 90th percentile.[2][6]

**Education and Employment Correlation**: Educated users and those in highly-paid professional occupations are substantially more likely to use ChatGPT for work-related purposes, indicating differential adoption patterns based on socioeconomic factors.[1][3][5]

### Behavioral Trends and User Engagement

All user cohorts followed similar patterns—flat usage through 2024 followed by substantial increases in early 2025, suggesting significant improvements in ChatGPT's capabilities or user-friendliness. Even early adopters from Q1 2023, who initially showed declining usage through 2024, experienced a 40% increase by July 2025.[2]

The data indicates that earlier sign-ups consistently maintain higher usage levels, but usage has grown within every cohort, attributed to both improvements in model capabilities and users gradually discovering new applications for existing features.[7]

## Most Common Use Cases and Taxonomy Classifications

### Primary Usage Categories

The research identifies three dominant use cases that collectively account for nearly 80% of all ChatGPT conversations: "Practical Guidance," "Seeking Information," and "Writing," representing about 77% of total usage.[1][3][5]

**Practical Guidance (29% of overall usage)**: This category has remained consistently at roughly 29% of overall usage and includes tutoring and teaching, how-to advice across various topics, and creative ideation. Within this category, 10.2% of all user messages (36% of Practical Guidance messages) are requests for tutoring or teaching, while another 8.5% of total messages (30% of Practical Guidance) involve general how-to advice on diverse topics.[5][7]

**Seeking Information (24% of usage)**: This category has grown significantly from 14% to 24% of all usage between July 2024 and July 2025. It includes searching for information about people, current events, products, and recipes, functioning as a close substitute for web search through conversational search that synthesizes sources.[5][7]

**Writing (24% overall, declining from 36%)**: Writing usage has declined from 36% of all usage in July 2024 to 24% a year later. This category encompasses automated production of emails, documents, and communications, as well as editing, critiquing, summarizing, and translating user-provided text. Notably, about two-thirds of all Writing messages ask ChatGPT to modify existing user text rather than creating new content from scratch.[5][7]

### Detailed Subcategory Analysis

**Writing Subcategories** break down as follows:
- Editing or Critiquing Provided Text: 10.6% of all messages
- Personal Writing or Communication: 8.0% of all messages  
- Translation: 4.5% of all messages
- Writing Fiction: 1.4% of all messages[5]

**Technical Help** has declined significantly from 12% to 5% of usage, including:
- Computer Programming: 4.2% of messages
- Mathematical Calculations: 3% of messages
- Data Analysis: 0.4% of messages[5]

**Other Notable Categories**:
- Multimedia: 6% of messages (grew from 2% to just over 7%, with a large spike in April 2025 after new image-generation capabilities were released)
- Self-Expression: 4.3% of messages
- Relationships and Personal Reflection: 1.9% of messages
- Games and Role Play: 0.4% of messages[5]

### Work-Related Usage Taxonomy

For work-related messages specifically, Writing dominates at approximately 40% of all work-related messages in July 2025, making it by far the most common work use case. Practical Guidance ranks second at 24% of work messages. Technical Help has declined from 18% of work-related messages in July 2024 to just over 10% in July 2025.[5]

Writing accounts for 42% of work-related messages overall and more than half of all messages for users in management and business occupations. This highlights the chatbot's unique ability to generate digital outputs compared to traditional search engines.[11]

### Intent-Based Classification System

The study introduces a novel taxonomy classifying messages by user intent:

**Asking (49% growing to 51.6%)**: Seeking information or advice to become better informed or make better decisions in work, school, or personal contexts. "Asking" messages—focused on seeking advice and judgment rather than task completion—have grown faster and consistently receive higher quality ratings from both automated classifiers and direct user feedback.[5][10]

**Doing (40% declining to 34.6%)**: Messages requesting ChatGPT to perform specific tasks for the user, such as drafting emails or writing code. These messages request output created primarily by the model. For work-related messages, Doing constitutes nearly 56% of work-related queries.[5]

**Expressing (11% growing to 13.8%)**: Statements that neither ask for information nor request task performance.[5]

### Professional Activity Mapping

Using O*NET work activity classifications, approximately 81% of work-related messages associate with two broad work activities: (1) obtaining, documenting, and interpreting information; and (2) making decisions, giving advice, solving problems, and thinking creatively. The work activities associated with ChatGPT usage show remarkable similarity across very different types of occupations.[5]

## Trends and Patterns from Data Analysis

### Work vs. Non-Work Usage Evolution

One of the most significant trends is the shift in work versus non-work usage patterns. While both categories have grown substantially, non-work messages have expanded much faster. Work and non-work-related messages both grew rapidly between June 2024 and June 2025, but non-work messages increased from 53% to 73% of total usage during this period.[1][3]

In absolute terms, work-related usage grew to 716 million messages per day (a 336% year-over-year increase), while non-work related usage exploded to 1,911 million messages per day (an 802% year-over-year increase). This shift represents changing usage patterns within existing user cohorts rather than compositional changes in new ChatGPT users.[7][8]

### Temporal Usage Pattern Changes

Message volume grew 5x between July 2024 and July 2025, with consistent growth across all user cohorts attributed to both model improvements and users discovering new capabilities.[7] The data shows all user cohorts following similar patterns—relatively flat usage through 2024 followed by substantial increases in early 2025.

### Topic Evolution Over Time

Several categories showed notable changes:
- **Seeking Information** grew dramatically from 14% to 24% between July 2024 and July 2025
- **Writing** declined from 36% to 24% over the same period
- **Practical Guidance** remained stable at roughly 29% of overall usage
- **Technical Help** declined significantly from 12% to around 5%
- **Multimedia** grew from 2% to just over 7%, with a large spike in April 2025 following new image-generation capabilities[5][7]

### Intent Pattern Evolution

The shift in user intent shows increasing sophistication in ChatGPT usage:
- **Asking** intent grew from roughly equal share with Doing in July 2024 to 51.6% by late June 2025
- **Doing** intent declined from roughly equal share to 34.6% by late June 2025  
- **Expressing** grew from just under 8% to 13.8% over the same period[5]

This trend toward "Asking" is particularly significant because these interactions consistently receive higher quality ratings, suggesting users are discovering more valuable ways to interact with the AI system.

### Geographic and Demographic Shifts

The democratization of ChatGPT usage is evident in geographic adoption patterns. Growth rates in the lowest income countries exceeded those in the highest income countries by more than 4x by May 2025. The most dramatic growth occurred in middle-income countries ($10,000–40,000 GDP-per-capita) when comparing May 2024 to May 2025 adoption rates.[6][7]

The complete closing of the gender gap represents another major demographic shift, with female users growing from 37% of users with classifiable names in January 2024 to 52% by July 2025.[6]

### User Satisfaction and Quality Trends

User satisfaction metrics show consistent improvement, with "good" interactions now 4x more common than bad ones. Positive interactions outnumber negative ones by approximately 4:1, and this ratio has remained stable as usage has scaled.[1] The data indicates that "Asking" questions yields the highest-rated outcomes, suggesting users are gravitating toward the most effective interaction patterns.

### Economic Impact Assessment

The research provides quantitative estimates of ChatGPT's economic value. The authors estimate that U.S. users would require roughly $98 compensation to give up generative AI for a month, implying at least $97 billion in annual consumer surplus in 2024 alone. This figure quantifies time saved, cost reductions, and improved information access.[10]

The researchers conclude that ChatGPT's strongest economic value lies in its role as a decision-support tool that generates tailored, actionable outputs mapping to common work activities across various jobs, distinguishing it fundamentally from traditional search engines. This economic value is especially pronounced in knowledge-intensive jobs where decision support capabilities are most valuable.[1][3]

### Future Implications

The data reveals that while economic analysis of AI has traditionally focused on productivity impacts in paid work, the impact on non-work activities (home production) appears to be on a similar or potentially larger scale. With over 70% of usage now non-work-related, ChatGPT's broader societal impact extends well beyond workplace productivity improvements, suggesting significant implications for how people access information, learn, and make decisions in their personal lives.[1][7]

### Sources

[1] How People Really Use ChatGPT: Findings from NBER Research: https://techmaniacs.com/2025/09/15/how-people-really-use-chatgpt-findings-from-nber-research/

[2] How People Use ChatGPT: https://forklightning.substack.com/p/how-people-use-chatgpt

[3] NBER Working Paper W34255: https://www.nber.org/papers/w34255

[4] SSRN Abstract 5487080: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080

[5] NBER Working Paper PDF: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf

[6] How People Are Using ChatGPT - OpenAI: https://openai.com/index/how-people-are-using-chatgpt/

[7] MBI Deep Dives Analysis: https://www.mbi-deepdives.com/how-people-use-chatgpt-and-its-implications-portfolio-change/

[8] David Deming LinkedIn Post: https://www.linkedin.com/posts/david-deming-8b93a2272_how-do-people-actually-use-chatgpt-to-find-activity-7374919341114290177-y_-F

[9] Ars Technica Analysis: https://arstechnica.com/ai/2025/09/seven-things-we-learned-from-openais-first-study-on-chatgpt-usage/

[10] Chosun English Coverage: https://www.chosun.com/english/industry-en/2025/09/16/O53VKIKLUVH3JKK7ZFWD3VYRXE/

[11] Agentic Partner Group LinkedIn: https://www.linkedin.com/posts/agentic-partner-group_economic-research-chatgpt-usage-paperpdf-activity-7374865459738968065-jD5s

[12] Saif Al Ramahi LinkedIn: https://www.linkedin.com/posts/saifalramahi_openai-just-revealed-how-700-million-people-activity-7373985297618927616-VZG3


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

In [23]:
#### CODE FOR ACTIVITY HERE

# Simple experiment: Try 3 different configurations and compare results
# 
# IMPORTANT: Due to Anthropic's rate limits (30k tokens/min), this uses a SIMPLER approach:
# - Uses FEWER iterations and searches to reduce token usage
# - Waits 2 MINUTES between experiments (120 seconds)
# - You may see "Summarization timed out" warnings - these are normal and handled

import asyncio
from IPython.display import Markdown, display

# Define 3 SIMPLIFIED experimental configurations (to avoid rate limits)
experiments = {
    "Experiment 1: Parallel (2 researchers)": {
        "max_concurrent_research_units": 2,  # 2 researchers (not 3 - saves tokens)
        "max_researcher_iterations": 1,      # Just 1 iteration
        "max_react_tool_calls": 2,           # Only 2 searches per researcher
    },
    "Experiment 2: Sequential Deep": {
        "max_concurrent_research_units": 1,  # Single researcher
        "max_researcher_iterations": 2,      # 2 iterations for depth
        "max_react_tool_calls": 2,           # 2 searches per researcher
    },
    "Experiment 3: Ultra Fast": {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 1,      # Minimal iterations
        "max_react_tool_calls": 1,           # Just 1 search
    },
}

# VERY simple research question to minimize token usage
simple_question = "What are 3 benefits of AI chatbots?"

async def run_experiment(name, settings):
    """Run a single experiment with specific settings."""
    print(f"\n{'='*80}")
    print(f"Running: {name}")
    print(f"Settings: {settings}")
    print(f"{'='*80}\n")
    
    # Create config for this experiment
    exp_config = {
        "configurable": {
            "research_model": "anthropic:claude-sonnet-4-20250514",
            "research_model_max_tokens": 10000,
            "compression_model": "anthropic:claude-sonnet-4-20250514",
            "compression_model_max_tokens": 8192,
            "final_report_model": "anthropic:claude-sonnet-4-20250514",
            "final_report_model_max_tokens": 10000,
            "summarization_model": "anthropic:claude-sonnet-4-20250514",
            "summarization_model_max_tokens": 8192,
            "allow_clarification": False,  # Skip clarification for faster testing
            "search_api": "tavily",
            "max_content_length": 50000,
            "thread_id": str(uuid.uuid4()),
            **settings  # Add experimental settings
        }
    }
    
    # Track timing
    import time
    start_time = time.time()
    
    final_report = None
    
    try:
        # Run the research
        async for event in graph.astream(
            {"messages": [{"role": "user", "content": simple_question}]},
            exp_config,
            stream_mode="updates"
        ):
            for node_name, node_output in event.items():
                if node_name == "final_report_generation" and "final_report" in node_output:
                    final_report = node_output["final_report"]
        
        elapsed_time = time.time() - start_time
        
        print(f"\n✓ Completed in {elapsed_time:.2f} seconds\n")
        
        # Display result summary
        if final_report:
            display(Markdown(f"### {name} - Result Summary"))
            display(Markdown(final_report[:800] + "...\n\n*(truncated for comparison)*\n"))
            print(f"Full report length: {len(final_report)} characters\n")
        
        return elapsed_time, len(final_report) if final_report else 0
    
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"\n❌ Experiment failed after {elapsed_time:.2f} seconds")
        print(f"Error: {str(e)[:200]}\n")
        return elapsed_time, 0

# Run all experiments
async def run_all_experiments():
    results = {}
    
    for i, (exp_name, exp_settings) in enumerate(experiments.items()):
        time_taken, report_length = await run_experiment(exp_name, exp_settings)
        results[exp_name] = {"time": time_taken, "length": report_length}
        
        # Add 2 minute delay between experiments to avoid rate limits
        if i < len(experiments) - 1:  # Don't wait after the last experiment
            wait_time = 120  # 2 minutes
            print(f"\n⏳ Waiting {wait_time} seconds (2 minutes) before next experiment...")
            print(f"   (This ensures we stay under Anthropic's 30k tokens/minute limit)")
            
            # Show countdown every 30 seconds
            for countdown in range(wait_time, 0, -30):
                print(f"   ⏰ {countdown} seconds remaining...")
                await asyncio.sleep(min(30, countdown))
            
            print(f"   ✓ Ready for next experiment!\n")
    
    # Display comparison as a table
    print("\n" + "="*90)
    print("EXPERIMENT COMPARISON TABLE")
    print("="*90)
    
    # Table header
    print(f"\n{'Experiment':<35} | {'Time (sec)':<12} | {'Report Length (chars)':<20}")
    print("-" * 90)
    
    # Table rows
    for exp_name, metrics in results.items():
        print(f"{exp_name:<35} | {metrics['time']:>10.2f}s | {metrics['length']:>18,} chars")
    
    print("-" * 90)
    
    # Summary statistics
    total_time = sum(m['time'] for m in results.values())
    avg_time = total_time / len(results) if results else 0
    total_chars = sum(m['length'] for m in results.values())
    avg_chars = total_chars / len(results) if results else 0
    
    print(f"{'AVERAGES':<35} | {avg_time:>10.2f}s | {avg_chars:>18,.0f} chars")
    print("=" * 90)
    
    print("\n" + "="*80)
    print("OBSERVATIONS:")
    print("="*80)
    print("""
1. Parallel researchers (Exp 1) - Using 2 researchers simultaneously can cover
   more ground but uses more tokens. Good for broad topics.

2. Sequential deep (Exp 2) - Single researcher with multiple iterations allows
   for follow-up questions and refinement. Better for depth.

3. Ultra fast (Exp 3) - Minimal configuration completes quickly but provides
   less comprehensive results. Good for simple questions.

Key Trade-offs: 
- Speed vs. Comprehensiveness: Faster configs give quicker but simpler results
- Token usage vs. Quality: More searches = better quality but higher API costs
- Parallel vs. Sequential: Parallel is faster but uses more tokens at once
    """)

# OPTION 1: Run just ONE experiment (SAFEST - recommended to start) - Uncomment line below:
# await run_experiment("Quick Test", experiments["Experiment 3: Ultra Fast"])

# OPTION 2: Run ALL 3 experiments (takes ~10-12 minutes with 2-min delays) - Uncomment line below:
await run_all_experiments()



Running: Experiment 1: Parallel (2 researchers)
Settings: {'max_concurrent_research_units': 2, 'max_researcher_iterations': 1, 'max_react_tool_calls': 2}


✓ Completed in 52.59 seconds



### Experiment 1: Parallel (2 researchers) - Result Summary

# Three Key Benefits of AI Chatbots: Comprehensive Analysis

AI chatbots have revolutionized how organizations interact with customers, streamline operations, and deliver services across multiple industries. Based on extensive research from technology publications, academic sources, and real-world case studies, three primary benefits consistently emerge as the most significant advantages of AI chatbot implementation.

## 24/7 Availability and Instant Response Capabilities

The most fundamental advantage of AI chatbots is their ability to provide continuous, round-the-clock service without human intervention. Unlike traditional customer service models that operate within business hours, AI chatbots maintain constant availability, responding to queries instantly at any time of day or night.
...

*(truncated for comparison)*


Full report length: 7890 characters


⏳ Waiting 120 seconds (2 minutes) before next experiment...
   (This ensures we stay under Anthropic's 30k tokens/minute limit)
   ⏰ 120 seconds remaining...
   ⏰ 90 seconds remaining...
   ⏰ 60 seconds remaining...
   ⏰ 30 seconds remaining...
   ✓ Ready for next experiment!


Running: Experiment 2: Sequential Deep
Settings: {'max_concurrent_research_units': 1, 'max_researcher_iterations': 2, 'max_react_tool_calls': 2}




✓ Completed in 201.20 seconds



...

# Three Key Benefits of AI Chatbots: Evidence-Based Analysis

AI chatbots have revolutionized how organizations interact with customers and streamline operations across industries. Based on extensive research from implementation studies, company reports, and academic analysis, three primary benefits consistently emerge as the most impactful and measurable advantages of AI chatbot deployment.

## 24/7 Availability and Instant Response Capability

One of the most significant advantages of AI chatbots is their ability to provide round-the-clock service without human intervention. Unlike traditional customer service models that operate within business hours, AI chatbots can handle inquiries, provide support, and engage with users at any time of day or night.

Research from MIT Technology Revie...

*(truncated for comparison)*


Full report length: 7457 characters


⏳ Waiting 120 seconds (2 minutes) before next experiment...
   (This ensures we stay under Anthropic's 30k tokens/minute limit)
   ⏰ 120 seconds remaining...
   ⏰ 90 seconds remaining...
   ⏰ 60 seconds remaining...
   ⏰ 30 seconds remaining...
   ✓ Ready for next experiment!


Running: Experiment 3: Ultra Fast
Settings: {'max_concurrent_research_units': 1, 'max_researcher_iterations': 1, 'max_react_tool_calls': 1}


✓ Completed in 57.27 seconds



### Experiment 3: Ultra Fast - Result Summary

# Three Key Benefits of AI Chatbots: Comprehensive Analysis

AI chatbots have revolutionized how organizations and individuals interact with technology, offering significant advantages across multiple domains. Based on extensive research and industry evidence, three distinct benefits stand out for their transformative impact: 24/7 availability and instant response capabilities, cost reduction and operational efficiency, and personalized user experiences with scalability.

## 24/7 Availability and Instant Response Capabilities

AI chatbots provide round-the-clock availability that human agents simply cannot match, delivering immediate responses to user inquiries regardless of time zones, holidays, or business hours. This continuous availability addresses a critical gap in traditional custom...

*(truncated for comparison)*


Full report length: 9157 characters


EXPERIMENT COMPARISON TABLE

Experiment                          | Time (sec)   | Report Length (chars)
------------------------------------------------------------------------------------------
Experiment 1: Parallel (2 researchers) |      52.59s |              7,890 chars
Experiment 2: Sequential Deep       |     201.20s |              7,457 chars
Experiment 3: Ultra Fast            |      57.27s |              9,157 chars
------------------------------------------------------------------------------------------
AVERAGES                            |     103.69s |              8,168 chars

OBSERVATIONS:

1. Parallel researchers (Exp 1) - Using 2 researchers simultaneously can cover
   more ground but uses more tokens. Good for broad topics.

2. Sequential deep (Exp 2) - Single researcher with multiple iterations allows
   for follow-up questions and refinement. Better for depth.

3. Ultra fast (Exp 3) - Minimal configuration completes quickly but 

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs